In [ ]:

import torch
import torch.nn.functional as F

In [ ]:
def swa_sdpa_strided(q, k, v, window_sizes: tuple[int, int] = (15, 16)):
    bwd_win, fwd_win = window_sizes
    win_size = 1 + bwd_win + fwd_win
    B, H, N, D = q.shape
    assert win_size <= N, "window must not exceed sequence length"

    # Pad along the sequence (dim=-2) so that boundary positions have
    # zero-filled neighbours instead of out-of-bounds reads.
    k_padded = F.pad(k, (0, 0, bwd_win, fwd_win))  # (B, H, N+W-1, D)
    v_padded = F.pad(v, (0, 0, bwd_win, fwd_win))

    # as_strided view: (B, H, N, W, D)
    # Each position i sees k_padded[:, :, i : i+W, :]
    new_shape = (B, H, N, win_size, D)
    new_stride = (
        k_padded.stride(0),
        k_padded.stride(1),
        k_padded.stride(2),
        k_padded.stride(2),
        k_padded.stride(3),
    )
    k_win = torch.as_strided(k_padded, size=new_shape, stride=new_stride).contiguous()
    v_win = torch.as_strided(v_padded, size=new_shape, stride=new_stride).contiguous()

    q_r = q.unsqueeze(-2)  # (B, H, N, 1, D)

    # boundary mask (additive, float)
    # For positions near the edges the zero-padding is invalid; mask
    # those entries to -inf so softmax ignores them.
    row_idx = torch.arange(N, device=q.device)
    win_offsets = torch.arange(-bwd_win, fwd_win + 1, device=q.device)
    abs_pos = row_idx.unsqueeze(-1) + win_offsets.unsqueeze(0)

    invalid = (abs_pos < 0) | (abs_pos >= N)  # True where out-of-bounds

    # Shape (N, 1, W) so it broadcasts over (B, H, N, 1, W) attention logits
    attn_bias = torch.zeros(N, 1, win_size, device=q.device, dtype=q.dtype)
    attn_bias.masked_fill_(invalid.unsqueeze(1), float("-inf"))

    # SDPA
    out = F.scaled_dot_product_attention(
        q_r, k_win, v_win,
        attn_mask=attn_bias,
    )
    return out.squeeze(-2)  # (B, H, N, D)


if __name__ == "__main__":
    from src.swa_torch_strided import swa_strided

    torch.manual_seed(0)
    B, H, N, D = 2, 4, 128, 64
    q = torch.randn(B, H, N, D, device="cuda", dtype=torch.float32)
    k = torch.randn(B, H, N, D, device="cuda", dtype=torch.float32)
    v = torch.randn(B, H, N, D, device="cuda", dtype=torch.float32)

    for bwd, fwd in [(15, 16), (0, 0), (31, 0), (0, 31), (7, 7)]:
        ref = swa_strided(q, k, v, window_sizes=(bwd, fwd))
        out = swa_sdpa_strided(q, k, v, window_sizes=(bwd, fwd))
        err = (ref - out).abs().max().item()
        print(f"window=({bwd:>2},{fwd:>2})  max-abs-error = {err:.2e}")